In [ ]:
import os
import shutil
import pydicom

# --- CONFIGURATION ---
source_path = '../Data/SQUARE_F/'

print(f"--- STARTING SMART CLEANING (SQUARE) ---")

def get_referenced_uid(rtstruct_path):
    try:
        dcm = pydicom.dcmread(rtstruct_path, stop_before_pixels=True, force=True)
        if 'ReferencedFrameOfReferenceSequence' in dcm:
            rfor = dcm.ReferencedFrameOfReferenceSequence[0]
            if 'RTReferencedStudySequence' in rfor:
                rstudy = rfor.RTReferencedStudySequence[0]
                if 'RTReferencedSeriesSequence' in rstudy:
                    return rstudy.RTReferencedSeriesSequence[0].SeriesInstanceUID
    except: return None
    return None

if os.path.exists(source_path):
    patient_folders = sorted([f for f in os.listdir(source_path) if os.path.isdir(os.path.join(source_path, f))])
    print(f"Found {len(patient_folders)} patients.")
    
    for i, patient_id in enumerate(patient_folders):
        patient_dir = os.path.join(source_path, patient_id)
        
        # 1. Find RTStruct
        rt_struct_path = None
        for root, dirs, files in os.walk(patient_dir):
            for f in files:
                if 'rs.' in f.lower() or (f.lower().startswith('rs') and '.dcm' in f.lower()):
                    rt_struct_path = os.path.join(root, f)
                    break
            if rt_struct_path: break
            
        if not rt_struct_path:
            print(f"[{i+1}] {patient_id}: [SKIP] No RTStruct found")
            continue

        # 2. Get Target UID
        target_uid = get_referenced_uid(rt_struct_path)
        if not target_uid:
            print(f"[{i+1}] {patient_id}: [SKIP] RTStruct has no UID")
            continue
            
        # 3. Create Clean Folders
        ct_dir = os.path.join(patient_dir, 'CT')
        struct_dir = os.path.join(patient_dir, 'Struct')
        
        if not os.path.exists(ct_dir): os.makedirs(ct_dir)
        if not os.path.exists(struct_dir): os.makedirs(struct_dir)
        
        # 4. Move Matching Files
        moved_count = 0
        for root, dirs, files in os.walk(patient_dir):
            
            if 'CT' in root or 'Struct' in root: continue
            
            for f in files:
                full_path = os.path.join(root, f)
                try:
                    dcm = pydicom.dcmread(full_path, stop_before_pixels=True, force=True)
                    
                    # If it matches the UID -> Move to CT
                    if dcm.SeriesInstanceUID == target_uid:
                        shutil.move(full_path, os.path.join(ct_dir, f))
                        moved_count += 1
                        
                    # If it is the RTStruct -> Move to Struct
                    elif full_path == rt_struct_path:
                        shutil.move(full_path, os.path.join(struct_dir, f))
                        
                except: continue
        
        print(f"[{i+1}] {patient_id}: Organized {moved_count} CT files.")

    print("\nSUCCESS! Square dataset cleaned.")
else:
    print("Source path not found.")

--- STARTING SMART CLEANING (SQUARE) ---
Found 121 patients.
[1] R130505087: Organized 0 CT files.
[2] R1406007291: Organized 0 CT files.
[3] R1508007367: Organized 0 CT files.
[4] R1701004331: Organized 0 CT files.
[5] R1702006414: Organized 200 CT files.
[6] R1707004823: Organized 183 CT files.
[7] R1710009613: Organized 200 CT files.
[8] R1803003706: Organized 176 CT files.
[9] R1807000334: Organized 0 CT files.
[10] R1807003460: Organized 216 CT files.
[11] R1807005275: Organized 147 CT files.
[12] R1807007064: Organized 201 CT files.
[13] R1808004062: Organized 160 CT files.
[14] R1810004752: Organized 200 CT files.
[15] R1810006592: Organized 152 CT files.
[16] R1810007735: Organized 184 CT files.
[17] R1811001641: Organized 185 CT files.
[18] R1903001183: Organized 148 CT files.
[19] R1903004587: Organized 163 CT files.
[20] R1905000571: Organized 169 CT files.
[21] R1905002341: Organized 187 CT files.
[22] R1905004532: Organized 197 CT files.
[23] R1907005733: Organized 160 CT 